In [2]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict


In [3]:
class BMIstate(TypedDict):
    weight_kg:float
    height_m:float
    bmi:float
    bmi_category:str

def bmi_cal(state:BMIstate)->BMIstate:
    weight = state['weight_kg']
    height = state['height_m']
    bmi = weight/(height**2)
    state['bmi'] = round(bmi,2)
    return state

def compare_bmi(state:BMIstate)->BMIstate:
    bmi = state['bmi']
    if bmi > 30:
        state['bmi_category']='obese'
    elif bmi <30:
        state['bmi_category']='ok-fit'
    elif bmi > 20:
        state['bmi_category']='fit'
    elif bmi > 10:
        state['bmi_category']='underweight'

    return state

In [7]:
#define your graph
workflow = StateGraph(BMIstate)

#define your nodes
workflow.add_node('bmi_cal',bmi_cal)
workflow.add_node('bmi_category',compare_bmi)

#connect edges
workflow.add_edge(START,'bmi_cal')
workflow.add_edge('bmi_cal','bmi_category')
workflow.add_edge('bmi_category',END)

checkpointer = InMemorySaver()

#compile graph
chain = workflow.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
chain.invoke({'height_m':1.74,'weight_kg':80}, config=config1)

{'weight_kg': 80, 'height_m': 1.74, 'bmi': 26.42, 'bmi_category': 'ok-fit'}

In [9]:
chain.get_state(config1)

StateSnapshot(values={'weight_kg': 80, 'height_m': 1.74, 'bmi': 26.42, 'bmi_category': 'ok-fit'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f138d12-c3c4-6bd4-8002-558732e95d87'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-15T13:44:06.666542+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f138d12-c3c1-603d-8001-1c586efd5292'}}, tasks=(), interrupts=())

In [ ]:
# StateSnapshot(values={'weight_kg': 80, 'height_m': 1.74, 'bmi': 26.42, 'bmi_category': 'ok-fit'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f138d12-c3c4-6bd4-8002-558732e95d87'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-15T13:44:06.666542+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f138d12-c3c1-603d-8001-1c586efd5292'}}, tasks=(), interrupts=())